# Discrete-Time Survival Model: Original BoL Features

This notebook is a first survival-style baseline next to logistic regression and gradient boosting. Instead of collapsing the outcome to one binary label only, it creates one row per person and survey wave and models the hazard of the first delinquency/contact event in that wave.

This is still an exploratory baseline. It uses a compact subset of the strongest original BoL features so the model is easy to run without extra survival libraries.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    plt = None
    print("Matplotlib unavailable:", exc)

try:
    from IPython.display import display as safe_display
except Exception:
    def safe_display(x):
        print(x)


def find_project_dir(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (
            (candidate / "nlsy79_child_youngadult_selected_crime_features.csv").exists()
            and (candidate / "Tabular_based_models").exists()
            and (candidate / "BoL approach").exists()
        ):
            return candidate
    raise FileNotFoundError("Start Jupyter inside the Bol_Crime project folder or one of its subfolders.")

PROJECT_DIR = find_project_dir()
TABULAR_DIR = PROJECT_DIR / "Tabular_based_models"
MODEL_DIR = TABULAR_DIR / "survival_model"
OUT_DIR = MODEL_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = PROJECT_DIR / "nlsy79_child_youngadult_selected_crime_features.csv"
FEATURE_INDEX_PATH = PROJECT_DIR / "BoL approach" / "metadata_examples" / "child_crime_broad_persistent_feature_index.csv"
TARGETS_PATH = TABULAR_DIR / "data" / "targets" / "nlsy79_temporal_delinquency_targets_2000_2020.csv"
LR_IMPORTANCE_PATH = TABULAR_DIR / "logistic_regression" / "outputs" / "l2_logistic_regression_permutation_importance.csv"

TARGET_ANY = "later_any_delinquency_contact_2000_2020"
FIRST_EVENT_YEAR = "later_delinquency_contact_first_event_year_2000_2020"
LAST_OBSERVED_YEAR = "later_delinquency_contact_last_observed_year_2000_2020"
MISSING_CODES = {-1, -2, -3, -4, -5, -7}
WAVES = np.array([2000, 2002, 2004, 2006, 2008, 2010, 2012, 2014, 2016, 2018, 2020])
RANDOM_SEED = 2026
TEST_SIZE = 0.30
FEATURE_LIMIT = 60
L2_STRENGTH = 1.0
LEARNING_RATE = 0.02
MAX_ITER = 1200
TOL = 1e-7

print("Project dir:", PROJECT_DIR)
print("Target file:", TARGETS_PATH)


## Load Data and Select Features

The survival baseline uses the same original BoL feature index as LR/GB. To keep the first attempt lightweight, it starts with the top 60 features from the LR permutation-importance output. If that file is unavailable, it falls back to the first 60 original BoL features.


In [ ]:
data = pd.read_csv(DATA_PATH)
feature_index = pd.read_csv(FEATURE_INDEX_PATH)
targets = pd.read_csv(TARGETS_PATH)

available_features = [c for c in feature_index["csv_code"].tolist() if c in data.columns]
if LR_IMPORTANCE_PATH.exists():
    lr_importance = pd.read_csv(LR_IMPORTANCE_PATH)
    selected_features = [c for c in lr_importance["csv_code"].tolist() if c in available_features][:FEATURE_LIMIT]
else:
    selected_features = available_features[:FEATURE_LIMIT]

feature_year = feature_index.set_index("csv_code")["survey_year"].to_dict()

person_df = data[["C0000100"] + selected_features].merge(
    targets[["C0000100", TARGET_ANY, FIRST_EVENT_YEAR, LAST_OBSERVED_YEAR]],
    on="C0000100",
    how="inner",
)
person_df = person_df[person_df[TARGET_ANY].notna() & person_df[LAST_OBSERVED_YEAR].notna()].copy()
person_df[TARGET_ANY] = person_df[TARGET_ANY].astype(int)

print("Eligible persons:", len(person_df))
print("Any-event base rate:", round(person_df[TARGET_ANY].mean(), 3))
print("Selected features:", len(selected_features))
feature_index[feature_index["csv_code"].isin(selected_features)][["csv_code", "ref_id", "variable", "survey_year", "feature_group", "question"]].head(15)


## Build Person-Wave Survival Table

Each person contributes one row for every observed wave while they are still at risk. If the first event occurs in a wave, that row is coded `event = 1` and later waves are not included for that person. If no event occurs, the person is treated as censored at the last observed wave.


In [ ]:
rows = []
for wave in WAVES:
    at_risk = (person_df[LAST_OBSERVED_YEAR] >= wave) & (
        person_df[FIRST_EVENT_YEAR].isna() | (person_df[FIRST_EVENT_YEAR] >= wave)
    )
    tmp = person_df.loc[at_risk, ["C0000100", TARGET_ANY, FIRST_EVENT_YEAR, LAST_OBSERVED_YEAR] + selected_features].copy()
    tmp["wave"] = int(wave)
    tmp["event"] = (tmp[FIRST_EVENT_YEAR] == wave).astype(int)
    rows.append(tmp)

survival_df = pd.concat(rows, ignore_index=True)
survival_df = survival_df.sort_values(["C0000100", "wave"]).reset_index(drop=True)

print("Person-wave rows:", len(survival_df))
print("Events:", int(survival_df["event"].sum()))
print("Event rate per person-wave:", round(survival_df["event"].mean(), 4))
print("Rows per wave:")
print(survival_df.groupby("wave")["event"].agg(rows="size", events="sum", event_rate="mean"))


## Preprocessing and Person-Level Train/Test Split

The split is done by person, not by row, so the same respondent cannot appear in both train and test. Features are only allowed if they were measured before the current wave. Negative NLSY missing codes are treated as missing and imputed from the training set.


In [ ]:
def stratified_person_split(person_ids, labels, test_size=0.30, seed=2026):
    rng = np.random.default_rng(seed)
    train_ids = []
    test_ids = []
    person_ids = np.asarray(person_ids)
    labels = np.asarray(labels)
    for label in sorted(np.unique(labels)):
        ids = person_ids[labels == label].copy()
        rng.shuffle(ids)
        n_test = int(round(len(ids) * test_size))
        test_ids.extend(ids[:n_test].tolist())
        train_ids.extend(ids[n_test:].tolist())
    return set(train_ids), set(test_ids)

train_ids, test_ids = stratified_person_split(
    person_df["C0000100"].astype(int).to_numpy(),
    person_df[TARGET_ANY].astype(int).to_numpy(),
    TEST_SIZE,
    RANDOM_SEED,
)

survival_df["split"] = np.where(survival_df["C0000100"].astype(int).isin(test_ids), "test", "train")

x_raw = survival_df[selected_features].copy()
for col in selected_features:
    x_raw[col] = pd.to_numeric(x_raw[col], errors="coerce")
    x_raw[col] = x_raw[col].replace([np.inf, -np.inf], np.nan)
    x_raw.loc[x_raw[col].isin(MISSING_CODES), col] = np.nan
    year = feature_year.get(col)
    if str(year) != "XRND":
        year_num = pd.to_numeric(pd.Series([year]), errors="coerce").iloc[0]
        if pd.notna(year_num):
            x_raw.loc[survival_df["wave"] <= year_num, col] = np.nan

train_mask = survival_df["split"].eq("train").to_numpy()
test_mask = survival_df["split"].eq("test").to_numpy()

medians = x_raw.loc[train_mask].median(axis=0, skipna=True).fillna(0.0)
x_imputed = x_raw.fillna(medians).to_numpy(dtype=float)

means = x_imputed[train_mask].mean(axis=0)
stds = x_imputed[train_mask].std(axis=0)
stds[stds == 0] = 1.0
x_std = np.clip(np.nan_to_num((x_imputed - means) / stds), -10, 10)

# Discrete-time survival needs a baseline hazard over time, so add one-hot wave dummies.
wave_dummies = pd.get_dummies(survival_df["wave"].astype(int), prefix="wave", drop_first=True).astype(float)
time_cols = wave_dummies.columns.tolist()
x_design_features = np.column_stack([x_std, wave_dummies.to_numpy(dtype=float)])
model_feature_names = selected_features + time_cols

y = survival_df["event"].astype(int).to_numpy()

x_train = x_design_features[train_mask]
x_test = x_design_features[test_mask]
y_train = y[train_mask]
y_test = y[test_mask]

print("Train person-wave rows:", len(y_train), "Test person-wave rows:", len(y_test))
print("Train event rate:", round(y_train.mean(), 4), "Test event rate:", round(y_test.mean(), 4))
print("Model columns:", x_train.shape[1])


## Fit Discrete-Time Hazard Model

This is a logistic hazard model: each row asks whether the first event happens in that wave, conditional on the person still being at risk. Year dummies act as a simple non-parametric baseline hazard.


In [ ]:
def sigmoid(z):
    z = np.clip(z, -35, 35)
    return 1.0 / (1.0 + np.exp(-z))


def logit(p):
    p = min(max(float(p), 1e-6), 1 - 1e-6)
    return float(np.log(p / (1 - p)))


def linear_predict(x_design, beta):
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        z = x_design @ beta
    return np.nan_to_num(z, nan=0.0, posinf=35.0, neginf=-35.0)


def auc_score(y_true, prob):
    order = np.argsort(prob)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(prob) + 1)
    pos = y_true == 1
    n_pos = int(pos.sum())
    n_neg = int((~pos).sum())
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    rank_sum_pos = ranks[pos].sum()
    return float((rank_sum_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


def fit_l2_logistic(x, y):
    x_design = np.column_stack([np.ones(len(x)), x])
    beta = np.zeros(x_design.shape[1], dtype=float)
    beta[0] = logit(y.mean())
    history = []
    for _ in range(MAX_ITER):
        p = sigmoid(linear_predict(x_design, beta))
        error = p - y
        with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
            grad = (x_design.T @ error) / len(y)
        grad = np.nan_to_num(grad, nan=0.0, posinf=0.0, neginf=0.0)
        grad[1:] += (L2_STRENGTH / len(y)) * beta[1:]
        new_beta = np.nan_to_num(beta - LEARNING_RATE * grad, nan=0.0, posinf=35.0, neginf=-35.0)
        eps = 1e-12
        loss = -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))
        loss += (L2_STRENGTH / (2 * len(y))) * np.sum(beta[1:] ** 2)
        history.append(float(loss))
        if np.max(np.abs(new_beta - beta)) < TOL:
            beta = new_beta
            break
        beta = new_beta
    return beta, history


def predict_prob(x, beta):
    x_design = np.column_stack([np.ones(len(x)), x])
    return sigmoid(linear_predict(x_design, beta))

surv_beta, surv_history = fit_l2_logistic(x_train, y_train)
train_hazard = predict_prob(x_train, surv_beta)
test_hazard = predict_prob(x_test, surv_beta)

print("Iterations:", len(surv_history))
print("Final train loss:", round(surv_history[-1], 4))
print("Row-level test AUC:", round(auc_score(y_test, test_hazard), 4))


## Evaluate Row-Level Hazard and Person-Level Cumulative Risk

The row-level output is the estimated hazard per wave. For a person-level comparison, hazards are combined into cumulative risk: `1 - product(1 - hazard across observed waves)`.


In [ ]:
def binary_metrics(y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    return {
        "threshold": threshold,
        "accuracy": float((pred == y_true).mean()),
        "auc": auc_score(y_true, prob),
        "sensitivity_tpr": tp / (tp + fn) if (tp + fn) else np.nan,
        "specificity_tnr": tn / (tn + fp) if (tn + fp) else np.nan,
        "predicted_positive_rate": float(pred.mean()),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }

pred_wave_df = survival_df.loc[test_mask, ["C0000100", "wave", "event", TARGET_ANY, FIRST_EVENT_YEAR, LAST_OBSERVED_YEAR]].copy()
pred_wave_df["hazard_probability"] = test_hazard
pred_wave_df["hazard_prediction_0_50"] = (pred_wave_df["hazard_probability"] >= 0.5).astype(int)

person_risk_df = (
    pred_wave_df
    .groupby("C0000100")
    .agg(
        y_true_any=(TARGET_ANY, "max"),
        first_event_year=(FIRST_EVENT_YEAR, "first"),
        last_observed_year=(LAST_OBSERVED_YEAR, "max"),
        n_waves=("wave", "count"),
        max_hazard=("hazard_probability", "max"),
    )
    .reset_index()
)
# Combine wave hazards into cumulative risk over the observed at-risk window.
risk_values = []
for child_id, g in pred_wave_df.groupby("C0000100"):
    hazards = np.clip(g["hazard_probability"].to_numpy(dtype=float), 0, 1)
    risk_values.append((child_id, float(1 - np.prod(1 - hazards))))
risk_df = pd.DataFrame(risk_values, columns=["C0000100", "cumulative_risk"])
person_risk_df = person_risk_df.merge(risk_df, on="C0000100", how="left")
person_risk_df["prediction_0_50"] = (person_risk_df["cumulative_risk"] >= 0.5).astype(int)
person_risk_df["correct_0_50"] = person_risk_df["prediction_0_50"] == person_risk_df["y_true_any"]

row_metrics = binary_metrics(y_test, test_hazard, threshold=0.5)
person_metrics = binary_metrics(
    person_risk_df["y_true_any"].astype(int).to_numpy(),
    person_risk_df["cumulative_risk"].to_numpy(dtype=float),
    threshold=0.5,
)

print("Row-level hazard metrics:")
print(pd.Series(row_metrics).to_string())
print("\nPerson-level cumulative-risk metrics:")
print(pd.Series(person_metrics).to_string())

pred_wave_df.to_csv(OUT_DIR / "discrete_time_survival_wave_predictions.csv", index=False)
person_risk_df.to_csv(OUT_DIR / "discrete_time_survival_person_risk_predictions.csv", index=False)
pd.DataFrame([
    {"level": "person_wave_hazard", **row_metrics},
    {"level": "person_cumulative_risk", **person_metrics},
]).to_csv(OUT_DIR / "discrete_time_survival_metrics.csv", index=False)

safe_display(person_risk_df.head(10))


## Model Coefficients

The coefficient table is a first interpretability check. Positive coefficients increase wave-level hazard; negative coefficients decrease it, conditional on the other selected features and the wave baseline.


In [ ]:
coef_df = pd.DataFrame({
    "model_feature": model_feature_names,
    "coefficient": surv_beta[1:],
})
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.merge(
    feature_index[["csv_code", "ref_id", "variable", "survey_year", "feature_group", "question"]],
    left_on="model_feature",
    right_on="csv_code",
    how="left",
)
coef_df = coef_df.sort_values("abs_coefficient", ascending=False)
coef_df.to_csv(OUT_DIR / "discrete_time_survival_coefficients.csv", index=False)

summary = {
    "model": "discrete_time_l2_logistic_survival_first_event",
    "target_event": FIRST_EVENT_YEAR,
    "person_level_target": TARGET_ANY,
    "feature_source": "original BoL broad persistent feature index; top LR permutation features for first attempt",
    "n_persons": int(len(person_df)),
    "n_person_wave_rows": int(len(survival_df)),
    "n_train_rows": int(len(y_train)),
    "n_test_rows": int(len(y_test)),
    "n_selected_features": int(len(selected_features)),
    "n_model_columns": int(x_train.shape[1]),
    "waves": WAVES.tolist(),
    "test_size": TEST_SIZE,
    "random_seed": RANDOM_SEED,
    "l2_strength": L2_STRENGTH,
    "learning_rate": LEARNING_RATE,
    "iterations": len(surv_history),
    "final_train_loss": surv_history[-1],
    "row_level_metrics": row_metrics,
    "person_level_metrics": person_metrics,
}
(OUT_DIR / "discrete_time_survival_summary.json").write_text(json.dumps(summary, indent=2))

safe_display(coef_df.head(15))

if plt is not None:
    top = coef_df.head(12).copy()
    labels = top["model_feature"].where(top["variable"].isna(), top["variable"].astype(str) + " (" + top["survey_year"].astype(str) + ")")
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(range(len(top)), top["abs_coefficient"])
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.set_xlabel("absolute coefficient")
    ax.set_title("Discrete-Time Survival: Top Coefficients")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "discrete_time_survival_top_coefficients.png", dpi=200, bbox_inches="tight")
    plt.show()
